## Модель лифта в здании
- В 5-этажном здании есть один лифт, вмещающий не более 6 человек.
- На вход этого здания на 1-м этаже поступает поток посетителей с интервалом от 1 до 20.
- Первый этаж – технический, и посетители на нем не задерживаются, а направляются на 2,3,4,5-й этажи, пользуясь лифтом.
- Посетитель, попав на свой этаж, находится на нем в течение случайного времени в интервале от 50 до 90.
- После этого он направляется к лифту, на нем опускается на 1-й этаж и покидает здание.
- Длительность перемещения лифта на один этаж равно 10. Длительность остановки лифта на этаже равно 12.

In [3]:
# -*- coding: utf-8 -*-
import random
import simpy
#
RANDOM_SEED = 526
MAX_LEVEL = 5



In [11]:

wait_times_log = []   # Для Задания 1 (очереди)
full_trips_count = 0  # Для Задания 2 (рейсы 1->5->1)
lifetimes_log = []    # Для Задания 4 (время жизни клиентов)

In [10]:
class Lift():
    def __init__(self, name, env):
        self.env = env  # ссылка на среду
        self.name = name  # имя лифта
        self.maxx = 6  # макс.емкость лифта
        self.curr_floor = 0  # этаж лифта
        self.going_up = True  # лифт едет вверх?
        self.Lift1Time = 10  # время между этажами
        self.Lift1Stop = 12  # время на этаже
        self.liftcab = simpy.Resource(self.env, capacity=self.maxx)  # ресурс
        # создаем словарь событий - этажи
        self.eventslift = {(i+1): env.event() for i in range(MAX_LEVEL)}

    def num_generator(self):
      while True:  # -1-2-3-4-3-2-
        i = 1
        while i <= MAX_LEVEL:
          yield int(i); i += 1
        i = MAX_LEVEL-1
        while i > 1:
          yield int(i); i -= 1

    def run(self):
        global  clients

        stage = self.num_generator()  # генератор - источник этажа
        yield self.env.timeout(5)  # подождем первых ездоков
        while len(clients) > 0:
            self.curr_floor = next(stage)  # лифт на этаже -использование генератора
            if (self.curr_floor in [1, MAX_LEVEL]):
                self.going_up = (self.curr_floor==1)  # лифт едет вверх?
            print(self.env.now, ":Ha_",self.curr_floor, " /",self.liftcab.count) # лифт приехал на этаж
            self.eventslift[self.curr_floor].succeed()  # создаем событие - на этаже!
            yield self.env.timeout(self.Lift1Stop)  # время остановки на этаже
            self.eventslift[self.curr_floor] = self.env.event()  # пересоздаем событие
            print(self.env.now, ":C_",self.curr_floor, " /",self.liftcab.count) # лифт уехал с этажа
            self.curr_floor = 0  # лифт не на этаже
            yield self.env.timeout(self.Lift1Time)  # время переезда на след.этаж
        else:
            print("That's all, folks! =", self.env.now)

In [5]:
class Client():
    def __init__(self, name, env):
        self.office_walk = random.randint(50,90) # время пребывания на этаже
        self.name = name   # имя клиента
        self.lift = env.lift  # ссылка на лифт
        self.env = env   # ссылка на среду
        self.gentime = self.env.now
        self.curr_floor = 1   # с какого этажа
        self.target_floor = random.randint(2,MAX_LEVEL)  # на какой этаж
        self.inside_ = False  # флаг - в лифте
        self.walking = False  # флаг - гуляю на этаже

    def run(self):
        global  clients
        while True:
            if not self.walking:
                yield self.lift.eventslift[self.curr_floor] # жду лифт на моем этаже
                #
                if (self.lift.going_up == (self.target_floor > self.curr_floor)) and \
                   (self.lift.liftcab.count < self.lift.maxx) :  # если лифт в нужном направлении и есть место
                    enterin = self.lift.liftcab.request() # занимаем ресурс
                    yield enterin
                    #
                    self.inside_ = True # клиент в лифте
                    yield self.env.timeout(1) # вход в лифт
                    print(self.env.now,">>",self.name, "!_",self.curr_floor, "!^",self.target_floor)
                    yield self.lift.eventslift[self.target_floor] # едем до нужного этажа
                    self.lift.liftcab.release(enterin)     # освобождаем ресурс
                    #
                    self.inside_ = False # клиент вышел из лифта
                    self.curr_floor = self.lift.curr_floor
                    self.walking = True # будет прогулка по делам в офисе
                    self.target_floor = 1 # и возвращение на 1 этаж
                    print(self.env.now,"<<",self.name, "!_",self.curr_floor, "!^",self.target_floor)
                    yield self.env.timeout(10)
                    if self.curr_floor==1:     # вернулся на 1й этаж?
                        clients.remove(self.name) # удаляем из числа клиентов
                        print(f"*{self.name}*: {self.env.now-self.gentime}: в офисе осталось {len(clients)} чел")
                        return
                else:
                    yield self.env.timeout(1) # надо подождать место в лифте
            else:
                yield self.env.timeout(self.office_walk) # прогулка по делам в офисе
                self.walking = False # прогулка закончена

In [6]:
def client_gen(env, qty_passengers):
  def char_generator():
      while True:  # ABC..Z+abc..z+0..9
        i = 65
        while i <= 90:
          yield chr(i); i += 1
        i = 97
        while i <= 122:
          yield chr(i); i += 1
        i = 57
        while i >= 48:
          yield chr(i); i -= 1

  global clients
  # создаем пассажиров
  nam = char_generator()
  for _ in range(qty_passengers):
    man = Client(next(nam), env)
    clients.append(man.name)
    env.process(man.run())
    yield env.timeout(random.randint(1,20))

In [7]:
env = simpy.Environment()
# список всех пассажиров
clients = []
#random.seed(RANDOM_SEED)

# создаем лифт
env.lift = Lift("_Lift_", env)
# подключаем процесс лифта
env.process(env.lift.run())
# подключаем процесс генерации клиентов
env.process(client_gen(env, 60))
# запускаем модель
env.run(until=2000)

5 :Ha_ 1  / 0
6 >> A !_ 1 !^ 4
17 :C_ 1  / 1
27 :Ha_ 2  / 1
39 :C_ 2  / 1
49 :Ha_ 3  / 1
61 :C_ 3  / 1
71 :Ha_ 4  / 1
71 << A !_ 4 !^ 1
83 :C_ 4  / 0
93 :Ha_ 5  / 0
105 :C_ 5  / 0
115 :Ha_ 4  / 0
127 :C_ 4  / 0
137 :Ha_ 3  / 0
149 :C_ 3  / 0
159 :Ha_ 2  / 0
171 :C_ 2  / 0
181 :Ha_ 1  / 0
182 >> B !_ 1 !^ 3
182 >> C !_ 1 !^ 5
182 >> D !_ 1 !^ 2
182 >> E !_ 1 !^ 4
182 >> F !_ 1 !^ 2
182 >> G !_ 1 !^ 2
193 :C_ 1  / 6
203 :Ha_ 2  / 6
203 << D !_ 2 !^ 1
203 << F !_ 2 !^ 1
203 << G !_ 2 !^ 1
215 :C_ 2  / 3
225 :Ha_ 3  / 3
225 << B !_ 3 !^ 1
237 :C_ 3  / 2
247 :Ha_ 4  / 2
247 << E !_ 4 !^ 1
259 :C_ 4  / 1
269 :Ha_ 5  / 1
269 << C !_ 5 !^ 1
281 :C_ 5  / 0
291 :Ha_ 4  / 0
292 >> A !_ 4 !^ 1
303 :C_ 4  / 1
313 :Ha_ 3  / 1
314 >> B !_ 3 !^ 1
325 :C_ 3  / 2
335 :Ha_ 2  / 2
336 >> F !_ 2 !^ 1
336 >> G !_ 2 !^ 1
336 >> D !_ 2 !^ 1
347 :C_ 2  / 5
357 :Ha_ 1  / 5
357 << A !_ 1 !^ 1
357 << B !_ 1 !^ 1
357 << F !_ 1 !^ 1
357 << G !_ 1 !^ 1
357 << D !_ 1 !^ 1
358 >> T !_ 1 !^ 3
358 >> b !_ 1 !^ 4
358 >> 

*Задание для самостоятельной работы:*
1) записать в журнал время ожидания лифта пассажирами в очередях на этажах
2) посчитать количество поездок лифта с 1 на 5 и обратно на 1 за время моделирования
3) скорректировать вход/выход клиентов в лифт - по очереди, а не одновременно
4) записать в журнал длительность жизни клиентов в модели

In [2]:
!pip install simpy

Класс Lift с отслеживанием полных рейсов и ресурсом двери

In [12]:
class Lift():
    def __init__(self, name, env):
        self.env = env  # ссылка на среду
        self.name = name  # имя лифта
        self.maxx = 6  # макс.емкость лифта
        self.curr_floor = 0  # этаж лифта
        self.going_up = True  # лифт едет вверх?
        self.Lift1Time = 10  # время между этажами
        self.Lift1Stop = 12  # время на этаже
        self.liftcab = simpy.Resource(self.env, capacity=self.maxx)  # ресурс вместимости кабины

        # ЗАДАНИЕ 3: Добавляем ресурс "дверь лифта" с емкостью 1.
        # Это заставит пассажиров входить и выходить строго по очереди (последовательно)
        self.door = simpy.Resource(self.env, capacity=1)

        # создаем словарь событий - этажи
        self.eventslift = {(i+1): env.event() for i in range(MAX_LEVEL)}

        # Переменная для Задания 2 (фиксация посещения 5 этажа перед возвратом на 1)
        self.has_reached_5 = False

    def num_generator(self):
        while True:  # Маршрут движения лифта по этажам: 1-2-3-4-5-4-3-2-1...
            i = 1
            while i <= MAX_LEVEL:
                yield int(i); i += 1
            i = MAX_LEVEL-1
            while i > 1:
                yield int(i); i -= 1

    def run(self):
        global clients, full_trips_count

        stage = self.num_generator()  # генератор - источник этажа
        yield self.env.timeout(5)  # подождем первых ездоков

        while len(clients) > 0:
            self.curr_floor = next(stage)  # лифт на этаже - использование генератора

            # ЗАДАНИЕ 2: Фиксируем полную поездку с 1 на 5 и обратно на 1
            if self.curr_floor == MAX_LEVEL:
                self.has_reached_5 = True
            elif self.curr_floor == 1 and self.has_reached_5:
                full_trips_count += 1
                self.has_reached_5 = False  # сбрасываем флаг для следующего цикла

            if (self.curr_floor in [1, MAX_LEVEL]):
                self.going_up = (self.curr_floor==1)  # лифт едет вверх?

            print(self.env.now, ":Ha_", self.curr_floor, " /", self.liftcab.count) # лифт приехал на этаж
            self.eventslift[self.curr_floor].succeed()  # создаем событие - на этаже!

            yield self.env.timeout(self.Lift1Stop)  # время остановки на этаже
            self.eventslift[self.curr_floor] = self.env.event()  # пересоздаем событие

            print(self.env.now, ":C_", self.curr_floor, " /", self.liftcab.count) # лифт уехал с этажа
            self.curr_floor = 0  # лифт не на этаже
            yield self.env.timeout(self.Lift1Time)  # время переезда на след.этаж
        else:
            print("That's all, folks! =", self.env.now)

Класс Client (Посетитель) с раздельным логированием и поочередным входом

In [13]:
class Client():
    def __init__(self, name, env):
        self.office_walk = random.randint(50,90) # время пребывания на этаже
        self.name = name   # имя клиента
        self.lift = env.lift  # ссылка на лифт
        self.env = env   # ссылка на среду
        self.gentime = self.env.now
        self.curr_floor = 1   # с какого этажа
        self.target_floor = random.randint(2,MAX_LEVEL)  # на какой этаж
        self.inside_ = False  # флаг - в лифте
        self.walking = False  # флаг - гуляю на этаже

    def run(self):
        global clients, wait_times_log, lifetimes_log
        while True:
            if not self.walking:
                # ЗАДАНИЕ 1: Запоминаем время начала ожидания лифта на этаже
                start_wait = self.env.now

                yield self.lift.eventslift[self.curr_floor] # жду лифт на моем этаже

                # если лифт идет в нужном направлении и в кабине есть место
                if (self.lift.going_up == (self.target_floor > self.curr_floor)) and \
                   (self.lift.liftcab.count < self.lift.maxx):

                    enterin = self.lift.liftcab.request() # занимаем место в кабине
                    yield enterin

                    # ЗАДАНИЕ 1: Вычисляем время ожидания и сохраняем в журнал
                    wait_duration = self.env.now - start_wait
                    wait_times_log.append((self.curr_floor, wait_duration))

                    self.inside_ = True # клиент в лифте

                    # ЗАДАНИЕ 3: Занимаем дверь лифта, чтобы войти по очереди
                    with self.lift.door.request() as req_door:
                        yield req_door
                        yield self.env.timeout(1) # вход в лифт занимает 1 единицу времени

                    print(self.env.now, ">>", self.name, "!_", self.curr_floor, "!^", self.target_floor)

                    yield self.lift.eventslift[self.target_floor] # едем до нужного этажа

                    # ЗАДАНИЕ 3: Занимаем дверь лифта, чтобы выйти по очереди
                    with self.lift.door.request() as req_door:
                        yield req_door
                        yield self.env.timeout(1) # выход из лифта занимает 1 единицу времени

                    self.lift.liftcab.release(enterin) # освобождаем место в кабине после выхода

                    self.inside_ = False # клиент вышел из лифта
                    self.curr_floor = self.lift.curr_floor
                    self.walking = True # будет прогулка по делам в офисе
                    self.target_floor = 1 # и последующее возвращение на 1 этаж
                    print(self.env.now, "<<", self.name, "!_", self.curr_floor, "!^", self.target_floor)

                    yield self.env.timeout(10)
                    if self.curr_floor == 1:     # вернулся на 1й этаж и покидает здание?
                        clients.remove(self.name) # удаляем из числа активных клиентов

                        # ЗАДАНИЕ 4: Записываем общую длительность нахождения клиента в системе
                        client_lifetime = self.env.now - self.gentime
                        lifetimes_log.append(client_lifetime)

                        print(f"*{self.name}*: {client_lifetime}: в офисе осталось {len(clients)} чел")
                        return
                else:
                    yield self.env.timeout(1) # лифт не подошел по условиям, ждем следующего такта
            else:
                yield self.env.timeout(self.office_walk) # прогулка по делам в офисе
                self.walking = False # прогулка закончена, идем обратно к лифту

In [14]:
def client_gen(env, qty_passengers):
    def char_generator():
        while True:  # Алфавитно-цифровой генератор имен клиентов: ABC..Z+abc..z+0..9
            i = 65
            while i <= 90:
                yield chr(i); i += 1
            i = 97
            while i <= 122:
                yield chr(i); i += 1
            i = 57
            while i >= 48:
                yield chr(i); i -= 1

    global clients
    nam = char_generator()
    for _ in range(qty_passengers):
        man = Client(next(nam), env)
        clients.append(man.name)
        env.process(man.run())
        yield env.timeout(random.randint(1, 20)) # Интервал прибытия новых посетителей в здание

In [15]:
# Настройка окружения
env = simpy.Environment()
clients = []
random.seed(RANDOM_SEED)

# Инициализируем/очищаем статистику
wait_times_log.clear()
lifetimes_log.clear()
full_trips_count = 0

# Создаем объект лифта и связываем его со средой
env.lift = Lift("_Lift_", env)

# Регистрируем параллельные процессы
env.process(env.lift.run())
env.process(client_gen(env, 60)) # Генерируем поток из 60 человек

# Запуск выполнения модели
env.run(until=2000)

5 :Ha_ 1  / 0
6 >> A !_ 1 !^ 3
7 >> B !_ 1 !^ 4
17 :C_ 1  / 2
27 :Ha_ 2  / 2
39 :C_ 2  / 2
49 :Ha_ 3  / 2
50 << A !_ 3 !^ 1
61 :C_ 3  / 1
71 :Ha_ 4  / 1
72 << B !_ 4 !^ 1
83 :C_ 4  / 0
93 :Ha_ 5  / 0
105 :C_ 5  / 0
115 :Ha_ 4  / 0
127 :C_ 4  / 0
137 :Ha_ 3  / 0
138 >> A !_ 3 !^ 1
149 :C_ 3  / 1
159 :Ha_ 2  / 1
171 :C_ 2  / 1
181 :Ha_ 1  / 1
182 << A !_ 1 !^ 1
183 >> C !_ 1 !^ 5
184 >> D !_ 1 !^ 3
185 >> E !_ 1 !^ 4
186 >> F !_ 1 !^ 4
187 >> G !_ 1 !^ 4
188 >> H !_ 1 !^ 3
*A*: 192: в офисе осталось 22 чел
193 :C_ 1  / 6
203 :Ha_ 2  / 6
215 :C_ 2  / 6
225 :Ha_ 3  / 6
226 << D !_ 3 !^ 1
227 << H !_ 3 !^ 1
237 :C_ 3  / 4
247 :Ha_ 4  / 4
248 << E !_ 4 !^ 1
249 << F !_ 4 !^ 1
250 << G !_ 4 !^ 1
259 :C_ 4  / 1
269 :Ha_ 5  / 1
270 << C !_ 5 !^ 1
281 :C_ 5  / 0
291 :Ha_ 4  / 0
292 >> B !_ 4 !^ 1
303 :C_ 4  / 1
313 :Ha_ 3  / 1
314 >> D !_ 3 !^ 1
315 >> H !_ 3 !^ 1
325 :C_ 3  / 3
335 :Ha_ 2  / 3
347 :C_ 2  / 3
357 :Ha_ 1  / 3
358 << B !_ 1 !^ 1
359 << D !_ 1 !^ 1
360 << H !_ 1 !^ 1
361 >> W !_ 1 

In [16]:


# 1. Результат Задания 2
print(f"1. Количество полных рейсов лифта (1 -> 5 -> 1): {full_trips_count}")
print("-" * 60)

# 2. Результат Задания 1
if wait_times_log:
    all_waits = [t for f, t in wait_times_log]
    print("2. Анализ времени ожидания лифта пассажирами на этажах:")
    print(f"   - Всего зафиксировано фактов ожидания: {len(all_waits)}")
    print(f"   - Среднее время ожидания:  {sum(all_waits)/len(all_waits):.2f} ед. времени")
    print(f"   - Максимальное время ожидания: {max(all_waits)} ед. времени")

    # Поэтажная статистика ожидания для более глубокого анализа
    for floor in range(1, MAX_LEVEL + 1):
        floor_waits = [t for f, t in wait_times_log if f == floor]
        if floor_waits:
            print(f"     * Среднее ожидание на {floor}-м этаже: {sum(floor_waits)/len(floor_waits):.2f} ед.")
else:
    print("2. Данные по времени ожидания не собраны.")
print("-" * 60)

# 3. Результат Задания 4
if lifetimes_log:
    print("3. Анализ длительности нахождения (жизни) клиентов в здании:")
    print(f"   - Всего успешно покинуло здание: {len(lifetimes_log)} чел.")
    print(f"   - Среднее время в системе:  {sum(lifetimes_log)/len(lifetimes_log):.2f} ед. времени")
    print(f"   - Максимальное время в системе: {max(lifetimes_log)} ед. времени")
else:
    print("3. Ни один клиент не успел полностью завершить цикл работы до остановки модели.")
print("="*60)

1. Количество полных рейсов лифта (1 -> 5 -> 1): 11
------------------------------------------------------------
2. Анализ времени ожидания лифта пассажирами на этажах:
   - Всего зафиксировано фактов ожидания: 117
   - Среднее время ожидания:  38.99 ед. времени
   - Максимальное время ожидания: 164 ед. времени
     * Среднее ожидание на 1-м этаже: 19.70 ед.
     * Среднее ожидание на 2-м этаже: 92.32 ед.
     * Среднее ожидание на 3-м этаже: 15.15 ед.
     * Среднее ожидание на 4-м этаже: 32.00 ед.
     * Среднее ожидание на 5-м этаже: 94.90 ед.
------------------------------------------------------------
3. Анализ длительности нахождения (жизни) клиентов в здании:
   - Всего успешно покинуло здание: 57 чел.
   - Среднее время в системе:  905.30 ед. времени
   - Максимальное время в системе: 1461 ед. времени
